In [5]:
import os 
import math 
import pathlib
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM ,AutoConfig
from datasets import load_dataset

from llmcompressor import oneshot 
from llmcompressor.modifiers.gptq import GPTQModifier

In [6]:
#download the Qwen3-0.6B model from hugging face and put it in folder named model

from pathlib import Path 
from huggingface_hub import snapshot_download 

MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_DIR = Path("models/Qwen3-0.6B")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading model...")


snapshot_download(
    repo_id = MODEL_ID,
    local_dir=str(MODEL_DIR),
)

print("Download Complete!")
print("Saved to:", MODEL_DIR.resolve())

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 335.02it/s]

Download Complete!
Saved to: C:\Users\Anil\Documents\Learning_Projects\local-llm-compression\models\Qwen3-0.6B


In [7]:
from pathlib import Path 
MODEL_DIR = Path("models/Qwen3-0.6B")
print(MODEL_DIR.exists())
print(list(MODEL_DIR.iterdir()))

True
[WindowsPath('models/Qwen3-0.6B/.cache'), WindowsPath('models/Qwen3-0.6B/.gitattributes'), WindowsPath('models/Qwen3-0.6B/config.json'), WindowsPath('models/Qwen3-0.6B/generation_config.json'), WindowsPath('models/Qwen3-0.6B/LICENSE'), WindowsPath('models/Qwen3-0.6B/merges.txt'), WindowsPath('models/Qwen3-0.6B/model.safetensors'), WindowsPath('models/Qwen3-0.6B/README.md'), WindowsPath('models/Qwen3-0.6B/tokenizer.json'), WindowsPath('models/Qwen3-0.6B/tokenizer_config.json'), WindowsPath('models/Qwen3-0.6B/vocab.json')]


In [8]:
config = AutoConfig.from_pretrained(MODEL_DIR)
#print the number of hidden layers
print(config.num_hidden_layers)
print(config.hidden_size)  #tells the dimensionality of the model's hidden representations
print(config.num_attention_heads) #tells the number of attention heads
print(config.head_dim) #tells the head dimension which is how many numerical features each attention head uses for q and k representations
print(config.num_key_value_heads)#how many key-value heads the model has 

#calucalte the GQA 
print(config.num_attention_heads // config.num_key_value_heads)

28
1024
16
128
8
2


In [9]:
text = "I am learning how large language model works"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokens = tokenizer.tokenize(text)
print(tokens)

#tokens to id 
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(token_ids)

#convert ids to PyTorch tensor
input_ids = torch.tensor([token_ids])

print(input_ids.shape)


['I', 'Ġam', 'Ġlearning', 'Ġhow', 'Ġlarge', 'Ġlanguage', 'Ġmodel', 'Ġworks']
[40, 1079, 6832, 1246, 3460, 4128, 1614, 4278]
torch.Size([1, 8])


In [10]:
#load model into CPU 
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16, device_map="cpu")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 677.32it/s]


In [11]:
embedding_layer = model.get_input_embeddings()
print(embedding_layer.weight.shape) #this tells us that the vocab size is 151,936 and numerical value per embedding is 1024

torch.Size([151936, 1024])


In [12]:
#lets perform the embedding lookup
embeddings = embedding_layer(input_ids)
print(embeddings.shape) #this tells us batch size of 1, sequence length of 8 and embedding dimension of 1024

torch.Size([1, 8, 1024])


In [13]:
#peek inside the first transformer block within the Qwen model 
print(model.model.layers[0])

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


These are learned linear transformations.For example here q_proj transforms each token's
1024 dimensional representations into a 2048 dimensional query vector.
But why 2048?
We know that :
Attention heads = 16 
Head dimension = 128 
Therefore: 16 X 128 = 2048 which explain the output of q_proj

The key and value projections are smaller becuase Qwen3 uses Grouped Query Attentions(GQA) with
eight key/value heads. so 8 X 128 = 1024.



In [14]:
#inspect the query projection weights
print(model.model.layers[0].self_attn.q_proj.weight.shape)
#linear-layer weights are as [out_features,in_features]

torch.Size([2048, 1024])


In [15]:
query_vectors = model.model.layers[0].self_attn.q_proj(embeddings)
print(query_vectors.shape)

torch.Size([1, 8, 2048])


In [16]:
query_heads = query_vectors.reshape(1,8,16,128)
print(query_heads.shape) #Batch is 1, sequence length is 8, attention heads is 16 and features per head is 128

torch.Size([1, 8, 16, 128])


In [17]:
key_vectors = model.model.layers[0].self_attn.k_proj(embeddings)
print(key_vectors.shape)

torch.Size([1, 8, 1024])


In [18]:
key_heads = key_vectors.reshape(1,8,8,128)
print(key_heads.shape)

torch.Size([1, 8, 8, 128])


In [19]:
value_vectors = model.model.layers[0].self_attn.v_proj(embeddings)
print(value_vectors.shape)

torch.Size([1, 8, 1024])


In [20]:
value_heads = value_vectors.reshape(1,8,8,128)
print(value_heads.shape)

torch.Size([1, 8, 8, 128])


In [21]:
#transpose the query tensor
query_heads = query_heads.transpose(1,2)
print(query_heads.shape)

torch.Size([1, 16, 8, 128])


In [22]:
#transpose the key tensor
key_heads = key_heads.transpose(1,2)
print(key_heads.shape)

torch.Size([1, 8, 8, 128])


In [23]:
#conceptually we want [Batch, Heads, Tokens, Features]
value_heads = value_heads.transpose(1,2)
print(value_heads.shape)

torch.Size([1, 8, 8, 128])


In [24]:
#To calculate attention using batched matrix multiplication we can repeat key head twice just to demonstrate attention
#real GQA doesn't need to repeat 
key_heads_expanded = key_heads.repeat_interleave(2,dim=1)
print(key_heads_expanded.shape)

torch.Size([1, 16, 8, 128])


In [25]:
#calculate the attention score 
attention_scores = torch.matmul(query_heads, key_heads_expanded.transpose(-2,-1))
print(attention_scores.shape)

scaled_scores = attention_scores / (128 ** 0.5)

torch.Size([1, 16, 8, 8])


In [26]:
#apply casual masking 
casual_mask = torch.tril(torch.ones(8,8,dtype=torch.bool))
print(casual_mask.shape)

masked_scores = scaled_scores.masked_fill(~casual_mask, float("-inf"))
print(masked_scores)


torch.Size([8, 8])
tensor([[[[ 2.1667e-03,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 1.7242e-03, -1.9455e-04,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 2.7771e-03,  2.0447e-03,  5.9509e-04,  ...,        -inf,
                  -inf,        -inf],
          ...,
          [ 2.2125e-03,  3.0365e-03, -4.6939e-07,  ..., -2.0447e-03,
                  -inf,        -inf],
          [ 3.9368e-03,  2.5635e-03,  1.2398e-04,  ...,  7.5912e-04,
            1.7700e-03,        -inf],
          [ 2.5635e-03,  2.1667e-03,  2.9755e-03,  ...,  1.8215e-04,
            2.9755e-03,  2.7008e-03]],

         [[ 4.2114e-03,        -inf,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 2.6093e-03,  3.2349e-03,        -inf,  ...,        -inf,
                  -inf,        -inf],
          [ 3.2196e-03,  2.3041e-03,  1.8997e-03,  ...,        -inf,
                  -inf,        -inf],
 

In [27]:
attention_weights = torch.softmax(masked_scores, dim =-1)
print(attention_weights[0,0,0,:])
print(attention_weights[0, 0, 1, :])


tensor([1., 0., 0., 0., 0., 0., 0., 0.], dtype=torch.bfloat16,
       grad_fn=<SelectBackward0>)
tensor([0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [28]:
value_heads_expanded = value_heads.repeat_interleave(2,dim=1)
print(value_heads_expanded.shape)

torch.Size([1, 16, 8, 128])


In [29]:
attention_output = torch.matmul(attention_weights, value_heads_expanded)
print(attention_output.shape)

torch.Size([1, 16, 8, 128])


In [30]:
attention_output = attention_output.transpose(1,2)
print(attention_output.shape)

attention_output = attention_output.reshape(1,8,2048)
print(attention_output.shape)

torch.Size([1, 8, 16, 128])
torch.Size([1, 8, 2048])


In [31]:
projected_output = model.model.layers[0].self_attn.o_proj(attention_output)
print(projected_output.shape)
print(model.model.layers[0].self_attn.o_proj.weight.shape)

torch.Size([1, 8, 1024])
torch.Size([1024, 2048])


In [32]:
#count the parameters programatically
print(model.model.layers[0].self_attn.o_proj.weight.numel())

2097152


In [33]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

596049920


In [34]:
#BF16 memory 
bf16_memory_gib = total_params * 2 /(1024 **3)
print(f"size of the model is {bf16_memory_gib} GB", )

size of the model is 1.1102294921875 GB


In [35]:
#check for GPU 
print(torch.cuda.is_available())

True


In [36]:
print(torch.__version__, torch.version.cuda)

2.14.1+cu132 13.2


In [37]:
! nvidia-smi

Sun Oct 25 19:53:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 617.42                 KMD Version: 617.42        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   39C    P8             19W /  320W |     254MiB /  16376MiB |     18%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [38]:
print(torch.__version__, torch.version.cuda)

2.14.1+cu132 13.2


In [39]:
print(torch.cuda.is_available())

True


In [40]:
import transformers; print(transformers.__version__)

5.20.2


In [41]:
print(type(model).__name__)

Qwen3ForCausalLM


In [42]:
print(next(model.parameters()).device)

cpu


In [43]:
#move the model to GPU 
model = model.to("cuda")
print(next(model.parameters()).device)

cuda:0


In [48]:
#move the input tensors to the gpu as well
input_ids.shape
input_ids = input_ids.to("cuda")
print(input_ids.device)


cuda:0


In [49]:
#before running the model, setting it to evaluation mode
model.eval() 

#stop recording operations for automatic differentiation in the current thread
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [51]:
outputs = model(input_ids)
print(outputs.logits.shape)

torch.Size([1, 8, 151936])


Inside the GPU
```
Input tokens [1, 8]
       ↓
Token embeddings
       ↓
28 Transformer layers
  ├── Q, K, V projections
  ├── Grouped Query Attention
  ├── Feed-forward network
  └── Normalization and residual connections
       ↓
Final normalization
       ↓
Language model head
       ↓
Logits [1, 8, 151936]
```

In [56]:
#to predict token 9, we need logits from the last input position

next_token_logits = outputs.logits[:,-1,:]
print(next_token_logits.shape)

next_token_id = torch.argmax(next_token_logits,dim=-1)

print(next_token_id)

torch.Size([1, 151936])
tensor([11], device='cuda:0')


In [57]:
#decode the predicted token 
print(repr(tokenizer.decode(next_token_id[0].item())))

','


```
Input: 8 tokens
       │
       ▼
┌───────────────────────────┐
│       Qwen3-0.6B          │
│       NVIDIA RTX 4080     │
│                           │
│  28 Transformer layers    │
│  Attention + MLP          │
└─────────────┬─────────────┘
              │
              ▼
    Logits [1, 151936]
              │
              ▼
       argmax(dim=-1)
              │
              ▼
         Token ID: 11
              │
              ▼
           Tokenizer
              │
              ▼
             ","
```

In [58]:
#inspect what was originally received by the model
print(repr(tokenizer.decode(input_ids[0].tolist())))

'I am learning how large language model works'


How Autoregressive generation works

```
Original input (8 tokens)
        │
        ▼
     Qwen3
        │
        ▼
 Predict token 9: ","
        │
        ▼
Append token 9 to input
        │
        ▼
New input (9 tokens)
        │
        ▼
     Qwen3
        │
        ▼
 Predict token 10
        │
       ...


```

In [59]:
#Add the predicted token to out input 
input_ids = torch.cat([input_ids,next_token_id.unsqueeze(-1)], dim=-1)
print(input_ids.shape)

torch.Size([1, 9])


In [61]:
outputs = model(input_ids)
next_token_logits = outputs.logits[:,-1,:]

print(next_token_logits.shape)

next_token_id = torch.argmax(next_token_logits,dim=-1)

print(next_token_id)

#decode the predicted token 
print(repr(tokenizer.decode(next_token_id[0].item())))

torch.Size([1, 151936])
tensor([323], device='cuda:0')
' and'


In [62]:
#Add the predicted token to out input 
input_ids = torch.cat([input_ids,next_token_id.unsqueeze(-1)], dim=-1)
print(input_ids.shape)

torch.Size([1, 10])


In [63]:
outputs = model(input_ids)
next_token_logits = outputs.logits[:,-1,:]

print(next_token_logits.shape)

next_token_id = torch.argmax(next_token_logits,dim=-1)

print(next_token_id)

#decode the predicted token 
print(repr(tokenizer.decode(next_token_id[0].item())))

torch.Size([1, 151936])
tensor([358], device='cuda:0')
' I'


In [64]:
#Add the predicted token to out input 
input_ids = torch.cat([input_ids,next_token_id.unsqueeze(-1)], dim=-1)
print(input_ids.shape)

torch.Size([1, 11])


In [65]:
outputs = model(input_ids)
next_token_logits = outputs.logits[:,-1,:]

print(next_token_logits.shape)

next_token_id = torch.argmax(next_token_logits,dim=-1)

print(next_token_id)

#decode the predicted token 
print(repr(tokenizer.decode(next_token_id[0].item())))

torch.Size([1, 151936])
tensor([1184], device='cuda:0')
' need'


In [66]:
input_ids = torch.cat([input_ids, next_token_id.unsqueeze(-1)], dim=-1)

In [ ]:
generated_ids = input_ids.clone()
max_new_tokens = 10 

for step in range(max_new_tokens):
    outputs = model(generated_ids)
    next_token_logits = outputs.logits[:,-1,:]
    next_token_id = torch.argmax(next_token_logits, dim=-1) #greedy decoding because argmax() always selects the token with highest logit
    generated_ids = torch.cat([generated_ids, next_token_id.unsqueeze(-1)],dim=-1)

print(generated_ids.shape)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

torch.Size([1, 22])
I am learning how large language model works, and I need to understand the concept of the model's training process
